# Configuration commune

Ce notebook est indépendant : il monte Google Drive, définit les chemins du projet, puis exécute uniquement la partie demandée.

Les datasets doivent déjà exister dans :

`/content/drive/MyDrive/Projet_Deep_Learning_Musique/data/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Projet_Deep_Learning_Musique"
os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

for folder in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Dossier projet :", PROJECT_DIR)

In [ ]:
# Installation de pretty_midi si nécessaire.
# Cette bibliothèque sert à lire les fichiers MIDI du dataset MAESTRO.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pretty_midi") is None:
    print("Installation de pretty_midi...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pretty_midi"])
else:
    print("pretty_midi est déjà installé.")


# Partie III — RNN / LSTM / GRU / Seq2Seq sur MAESTRO MIDI

In [ ]:
import os
import glob
import time
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.model_selection import train_test_split

try:
    import pretty_midi
except ImportError:
    raise ImportError(
        "La bibliothèque pretty_midi n'est pas installée. "
        "Dans Colab, installe-la avec : !pip install pretty_midi"
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

required_vars = ["PROJECT_DIR", "DATA_DIR", "MODEL_DIR", "RESULTS_DIR"]
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise NameError(
        "Les variables suivantes doivent déjà exister dans le notebook : "
        + ", ".join(missing_vars)
    )

if "MAESTRO_DIR" not in globals():
    MAESTRO_DIR = os.path.join(DATA_DIR, "maestro")

print("Dossier MAESTRO :", MAESTRO_DIR)

## 2. Vérification du dataset MAESTRO

In [ ]:
if not os.path.exists(MAESTRO_DIR):
    raise FileNotFoundError(f"Le dossier MAESTRO est introuvable : {MAESTRO_DIR}")

midi_files = sorted(
    glob.glob(os.path.join(MAESTRO_DIR, "**", "*.mid"), recursive=True)
    + glob.glob(os.path.join(MAESTRO_DIR, "**", "*.midi"), recursive=True)
)

print("Nombre total de fichiers MIDI trouvés :", len(midi_files))

print("\nExemples de fichiers :")
for path in midi_files[:10]:
    print("-", path)

available_years = sorted([
    item for item in os.listdir(MAESTRO_DIR)
    if os.path.isdir(os.path.join(MAESTRO_DIR, item))
])

print("\nDossiers disponibles dans MAESTRO :")
print(available_years)

if len(midi_files) == 0:
    raise FileNotFoundError("Aucun fichier .mid ou .midi trouvé dans MAESTRO_DIR.")

## 3. Paramètres du mode rapide

In [ ]:
FAST_MODE = True
MAX_MIDI_FILES = 200

if FAST_MODE:
    midi_files_selected = midi_files[:MAX_MIDI_FILES]
else:
    midi_files_selected = midi_files

print("FAST_MODE :", FAST_MODE)
print("Nombre de fichiers MIDI sélectionnés :", len(midi_files_selected))

## 4. Extraction des notes MIDI

In [ ]:
def extract_notes_from_midi(midi_path, min_notes=30):
    """
    Extrait une séquence simple de hauteurs MIDI.
    Chaque note est représentée par son pitch entre 0 et 127.
    """
    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)

        notes = []

        for instrument in midi_data.instruments:
            if instrument.is_drum:
                continue

            for note in instrument.notes:
                notes.append((note.start, note.pitch))

        notes = sorted(notes, key=lambda x: x[0])
        pitches = [pitch for _, pitch in notes]

        if len(pitches) < min_notes:
            return None

        return pitches

    except Exception:
        return None


def build_note_corpus(midi_files):
    sequences = []
    used_files = 0
    ignored_files = 0
    total_notes = 0

    for i, midi_path in enumerate(midi_files):
        if (i + 1) % 25 == 0:
            print(f"Traitement : {i + 1}/{len(midi_files)} fichiers")

        sequence = extract_notes_from_midi(midi_path)

        if sequence is None:
            ignored_files += 1
            continue

        sequences.append(sequence)
        used_files += 1
        total_notes += len(sequence)

    mean_length = total_notes / used_files if used_files > 0 else 0

    print("\nExtraction terminée.")
    print("Fichiers utilisés :", used_files)
    print("Fichiers ignorés :", ignored_files)
    print("Nombre total de notes extraites :", total_notes)
    print("Longueur moyenne des séquences :", round(mean_length, 2))

    return sequences

## 5. Cache des séquences prétraitées

In [ ]:
cache_suffix = "fast200" if FAST_MODE else "full"
maestro_cache_path = os.path.join(RESULTS_DIR, f"maestro_note_sequences_{cache_suffix}.pt")

if os.path.exists(maestro_cache_path):
    print("Chargement du cache :", maestro_cache_path)

    cache_data = torch.load(
        maestro_cache_path,
        map_location="cpu",
        weights_only=False
    )

    note_sequences = cache_data["note_sequences"]

else:
    print("Aucun cache trouvé. Extraction des notes MIDI...")

    note_sequences = build_note_corpus(midi_files_selected)

    torch.save(
        {
            "note_sequences": note_sequences,
            "fast_mode": FAST_MODE,
            "max_midi_files": MAX_MIDI_FILES
        },
        maestro_cache_path
    )

    print("Cache sauvegardé :", maestro_cache_path)

if len(note_sequences) == 0:
    raise RuntimeError("Aucune séquence MIDI valide n'a été extraite.")

total_notes = sum(len(seq) for seq in note_sequences)
print("Nombre de séquences :", len(note_sequences))
print("Nombre total de notes :", total_notes)

## 6. Création du vocabulaire

In [ ]:
PAD_TOKEN = "PAD"
SOS_TOKEN = "SOS"
EOS_TOKEN = "EOS"

all_notes = sorted(set(note for seq in note_sequences for note in seq))

idx_to_note = {
    0: PAD_TOKEN,
    1: SOS_TOKEN,
    2: EOS_TOKEN
}

for i, note in enumerate(all_notes, start=3):
    idx_to_note[i] = note

note_to_idx = {note: idx for idx, note in idx_to_note.items()}

PAD_IDX = note_to_idx[PAD_TOKEN]
SOS_IDX = note_to_idx[SOS_TOKEN]
EOS_IDX = note_to_idx[EOS_TOKEN]

vocab_size = len(note_to_idx)

print("Taille du vocabulaire :", vocab_size)
print("PAD_IDX :", PAD_IDX)
print("SOS_IDX :", SOS_IDX)
print("EOS_IDX :", EOS_IDX)

print("\nExemples du mapping :")
for idx in list(idx_to_note.keys())[:15]:
    print(idx, "->", idx_to_note[idx])

## 7. Dataset pour prédiction de la prochaine note

In [ ]:
SEQ_LEN = 50
STRIDE = 25

class NextNoteDataset(Dataset):
    def __init__(self, note_sequences, note_to_idx, seq_len=50, stride=25):
        self.examples = []
        self.seq_len = seq_len
        self.note_to_idx = note_to_idx

        for seq in note_sequences:
            encoded = [note_to_idx[note] for note in seq if note in note_to_idx]

            if len(encoded) < 2:
                continue

            for start in range(0, len(encoded) - 1, stride):
                input_seq = encoded[start:start + seq_len]
                target_seq = encoded[start + 1:start + seq_len + 1]

                if len(input_seq) < seq_len:
                    input_seq = input_seq + [PAD_IDX] * (seq_len - len(input_seq))

                if len(target_seq) < seq_len:
                    target_seq = target_seq + [PAD_IDX] * (seq_len - len(target_seq))

                self.examples.append((input_seq, target_seq))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        x, y = self.examples[idx]

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )


next_note_dataset = NextNoteDataset(
    note_sequences=note_sequences,
    note_to_idx=note_to_idx,
    seq_len=SEQ_LEN,
    stride=STRIDE
)

print("Nombre total d'exemples :", len(next_note_dataset))

if len(next_note_dataset) == 0:
    raise RuntimeError("Le dataset de séquences est vide.")

In [ ]:
train_size = int(0.70 * len(next_note_dataset))
val_size = int(0.15 * len(next_note_dataset))
test_size = len(next_note_dataset) - train_size - val_size

train_dataset_seq, val_dataset_seq, test_dataset_seq = random_split(
    next_note_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

batch_size_seq = 64

train_loader_seq = DataLoader(train_dataset_seq, batch_size=batch_size_seq, shuffle=True)
val_loader_seq = DataLoader(val_dataset_seq, batch_size=batch_size_seq, shuffle=False)
test_loader_seq = DataLoader(test_dataset_seq, batch_size=batch_size_seq, shuffle=False)

print("Train :", len(train_dataset_seq))
print("Validation :", len(val_dataset_seq))
print("Test :", len(test_dataset_seq))
print("Batch size :", batch_size_seq)

## 8. Rappels théoriques courts

Un modèle de langage musical cherche à prédire la probabilité de la prochaine note à partir des notes précédentes.

Une séquence peut être factorisée avec la règle de chaîne :  
P(x1, x2, ..., xT) = P(x1) P(x2|x1) ... P(xT|x1, ..., xT-1)

La perplexité mesure à quel point le modèle est incertain. Une perplexité plus faible indique généralement une meilleure prédiction.

Le BPTT, ou Backpropagation Through Time, consiste à rétropropager l'erreur à travers les pas de temps d'un réseau récurrent.

Dans un RNN simple, les gradients peuvent disparaître ou exploser quand les séquences sont longues.

Les LSTM et GRU sont plus stables car ils utilisent des mécanismes de portes pour mieux contrôler la mémoire.

Le gradient clipping limite la norme des gradients pour éviter les mises à jour trop grandes.

Un modèle Seq2Seq encode une séquence d'entrée puis décode une séquence de sortie.

Le teacher forcing consiste à donner au décodeur la vraie note précédente pendant l'entraînement.

Le greedy decoding choisit à chaque étape la note la plus probable.

Le beam search garde plusieurs hypothèses de génération et choisit la meilleure séquence globale parmi elles.

## 9. Modèles RNN, LSTM et GRU

In [ ]:
class MusicRNNModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=2,
        rnn_type="LSTM",
        dropout=0.3
    ):
        super().__init__()

        self.rnn_type = rnn_type.upper()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX
        )

        if self.rnn_type == "RNN":
            self.rnn = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout
            )
        elif self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout
            )
        elif self.rnn_type == "GRU":
            self.rnn = nn.GRU(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout
            )
        else:
            raise ValueError("rnn_type doit être RNN, LSTM ou GRU.")

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        logits = self.fc(output)

        return logits, hidden

## 10. Fonctions d'entraînement et d'évaluation

In [ ]:
def calculate_perplexity(loss):
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")


def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def train_one_epoch_sequence(model, dataloader, criterion, optimizer, device, max_norm=1.0):
    model.train()

    running_loss = 0.0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, _ = model(x)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)

        optimizer.step()

        running_loss += loss.item() * x.size(0)

    return running_loss / len(dataloader.dataset)


def evaluate_sequence(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits, _ = model(x)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            running_loss += loss.item() * x.size(0)

    avg_loss = running_loss / len(dataloader.dataset)
    perplexity = calculate_perplexity(avg_loss)

    return avg_loss, perplexity

In [ ]:
def plot_sequence_curves(history, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train loss")
    plt.plot(epochs, history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["val_perplexity"], label="Validation perplexity")
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title("Perplexité")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

    print("Courbes sauvegardées :", save_path)


def fit_sequence_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=15,
    patience=4,
    save_path=None
):
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": []
    }

    best_val_loss = np.inf
    best_epoch = 0
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch_sequence(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_perplexity = evaluate_sequence(
            model,
            val_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_perplexity"].append(val_perplexity)

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val perplexity: {val_perplexity:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0

            if save_path is not None:
                torch.save(model.state_dict(), save_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping après {epoch} epochs.")
            break

    training_time = time.time() - start_time

    return history, best_val_loss, best_epoch, training_time

## 11. Comparaison RNN simple vs LSTM vs GRU

In [ ]:
sequence_experiments = [
    {"name": "RNN_simple", "rnn_type": "RNN"},
    {"name": "LSTM", "rnn_type": "LSTM"},
    {"name": "GRU", "rnn_type": "GRU"}
]

criterion_seq = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
learning_rate_seq = 0.001
num_epochs_seq = 15
patience_seq = 4

sequence_results = []

for experiment in sequence_experiments:
    model_name = experiment["name"]
    rnn_type = experiment["rnn_type"]

    print("\n" + "=" * 70)
    print("Expérience :", model_name)
    print("=" * 70)

    model_seq = MusicRNNModel(
        vocab_size=vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=2,
        rnn_type=rnn_type,
        dropout=0.3
    ).to(device)

    optimizer_seq = optim.Adam(model_seq.parameters(), lr=learning_rate_seq)

    model_save_path = os.path.join(MODEL_DIR, f"{model_name}_best.pt")
    curve_save_path = os.path.join(RESULTS_DIR, f"{model_name}_curves.png")

    history, best_val_loss, best_epoch, training_time = fit_sequence_model(
        model=model_seq,
        train_loader=train_loader_seq,
        val_loader=val_loader_seq,
        criterion=criterion_seq,
        optimizer=optimizer_seq,
        device=device,
        num_epochs=num_epochs_seq,
        patience=patience_seq,
        save_path=model_save_path
    )

    plot_sequence_curves(history, curve_save_path)

    sequence_results.append({
        "model_name": model_name,
        "rnn_type": rnn_type,
        "final_train_loss": history["train_loss"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_perplexity": history["val_perplexity"][-1],
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "trainable_parameters": count_trainable_parameters(model_seq),
        "training_time_seconds": training_time,
        "model_path": model_save_path,
        "curve_path": curve_save_path
    })

sequence_results_df = pd.DataFrame(sequence_results)

sequence_results_csv_path = os.path.join(RESULTS_DIR, "sequence_models_results.csv")
sequence_results_df.to_csv(sequence_results_csv_path, index=False)

print("Tableau comparatif sauvegardé :", sequence_results_csv_path)
display(sequence_results_df)

## 12. Génération musicale simple

In [ ]:
def midi_pitch_to_name(pitch):
    if isinstance(pitch, str):
        return pitch

    note_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    octave = pitch // 12 - 1
    note = note_names[pitch % 12]

    return f"{note}{octave}"


def tokens_to_midi_notes(tokens, idx_to_note):
    notes = []

    for token in tokens:
        note = idx_to_note.get(int(token), PAD_TOKEN)

        if note in [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN]:
            continue

        notes.append(note)

    return notes


def generate_sequence(model, seed_sequence, max_length=50, strategy="greedy", beam_width=3):
    model.eval()

    if strategy == "greedy":
        generated = list(seed_sequence)

        with torch.no_grad():
            for _ in range(max_length):
                x = torch.tensor(generated[-SEQ_LEN:], dtype=torch.long).unsqueeze(0).to(device)
                logits, _ = model(x)

                next_token = torch.argmax(logits[0, -1]).item()

                if next_token == EOS_IDX:
                    break

                generated.append(next_token)

        return generated

    elif strategy == "beam":
        beams = [(list(seed_sequence), 0.0)]

        with torch.no_grad():
            for _ in range(max_length):
                candidates = []

                for seq, score in beams:
                    x = torch.tensor(seq[-SEQ_LEN:], dtype=torch.long).unsqueeze(0).to(device)
                    logits, _ = model(x)

                    log_probs = torch.log_softmax(logits[0, -1], dim=0)
                    top_values, top_indices = torch.topk(log_probs, beam_width)

                    for log_prob, token in zip(top_values, top_indices):
                        new_seq = seq + [token.item()]
                        new_score = score + log_prob.item()
                        candidates.append((new_seq, new_score))

                beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]

        return beams[0][0]

    else:
        raise ValueError("strategy doit être 'greedy' ou 'beam'.")

In [ ]:
best_sequence_index = sequence_results_df.sort_values(
    by="best_val_loss",
    ascending=True
).index[0]

best_sequence_row = sequence_results_df.loc[best_sequence_index]

print("Meilleur modèle séquentiel :")
display(best_sequence_row)

best_sequence_model = MusicRNNModel(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    rnn_type=best_sequence_row["rnn_type"],
    dropout=0.3
).to(device)

best_sequence_model.load_state_dict(
    torch.load(best_sequence_row["model_path"], map_location=device)
)

test_x, test_y = test_dataset_seq[0]
seed_tokens = test_x[:10].tolist()

manual_notes = [60, 64, 67, 72]
manual_seed_tokens = [
    note_to_idx[note]
    for note in manual_notes
    if note in note_to_idx
]

if len(manual_seed_tokens) == 0:
    manual_seed_tokens = seed_tokens

greedy_tokens = generate_sequence(
    best_sequence_model,
    seed_tokens,
    max_length=30,
    strategy="greedy"
)

beam_tokens = generate_sequence(
    best_sequence_model,
    seed_tokens,
    max_length=30,
    strategy="beam",
    beam_width=3
)

manual_greedy_tokens = generate_sequence(
    best_sequence_model,
    manual_seed_tokens,
    max_length=30,
    strategy="greedy"
)

print("Seed test MIDI :")
print(tokens_to_midi_notes(seed_tokens, idx_to_note))

print("\nGénération greedy :")
print(tokens_to_midi_notes(greedy_tokens, idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(greedy_tokens, idx_to_note)])

print("\nGénération beam search :")
print(tokens_to_midi_notes(beam_tokens, idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(beam_tokens, idx_to_note)])

print("\nSeed manuel :")
print(tokens_to_midi_notes(manual_seed_tokens, idx_to_note))

print("\nGénération depuis seed manuel :")
print(tokens_to_midi_notes(manual_greedy_tokens, idx_to_note))

## 13. Mini système Seq2Seq

In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, next_note_dataset, half_len=25):
        self.examples = []
        self.half_len = half_len

        for x, y in next_note_dataset:
            full_seq = x.tolist()

            encoder_input = full_seq[:half_len]
            decoder_target = full_seq[half_len:half_len * 2]

            if len(decoder_target) < half_len:
                decoder_target = decoder_target + [PAD_IDX] * (half_len - len(decoder_target))

            decoder_input = [SOS_IDX] + decoder_target[:-1]

            self.examples.append((
                torch.tensor(encoder_input, dtype=torch.long),
                torch.tensor(decoder_input, dtype=torch.long),
                torch.tensor(decoder_target, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


seq2seq_dataset = Seq2SeqDataset(next_note_dataset, half_len=25)

train_size_s2s = int(0.70 * len(seq2seq_dataset))
val_size_s2s = int(0.15 * len(seq2seq_dataset))
test_size_s2s = len(seq2seq_dataset) - train_size_s2s - val_size_s2s

train_dataset_s2s, val_dataset_s2s, test_dataset_s2s = random_split(
    seq2seq_dataset,
    [train_size_s2s, val_size_s2s, test_size_s2s],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader_s2s = DataLoader(train_dataset_s2s, batch_size=64, shuffle=True)
val_loader_s2s = DataLoader(val_dataset_s2s, batch_size=64, shuffle=False)
test_loader_s2s = DataLoader(test_dataset_s2s, batch_size=64, shuffle=False)

print("Seq2Seq train :", len(train_dataset_s2s))
print("Seq2Seq validation :", len(val_dataset_s2s))
print("Seq2Seq test :", len(test_dataset_s2s))

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)

        return outputs, hidden


class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_token, hidden):
        embedded = self.embedding(input_token)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(output)

        return logits, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, decoder_input, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        target_len = decoder_input.size(1)
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, target_len, vocab_size).to(self.device)

        _, hidden = self.encoder(src)

        input_token = decoder_input[:, 0].unsqueeze(1)

        for t in range(target_len):
            logits, hidden = self.decoder(input_token, hidden)
            outputs[:, t:t+1, :] = logits

            teacher_force = random.random() < teacher_forcing_ratio
            top_token = logits.argmax(dim=-1)

            if t + 1 < target_len:
                input_token = decoder_input[:, t + 1].unsqueeze(1) if teacher_force else top_token

        return outputs

In [ ]:
def train_seq2seq(model, dataloader, criterion, optimizer, device, teacher_forcing_ratio=0.5):
    model.train()

    running_loss = 0.0

    for src, decoder_input, target in dataloader:
        src = src.to(device)
        decoder_input = decoder_input.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        logits = model(src, decoder_input, teacher_forcing_ratio=teacher_forcing_ratio)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * src.size(0)

    return running_loss / len(dataloader.dataset)


def evaluate_seq2seq(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for src, decoder_input, target in dataloader:
            src = src.to(device)
            decoder_input = decoder_input.to(device)
            target = target.to(device)

            logits = model(src, decoder_input, teacher_forcing_ratio=0.0)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target.reshape(-1)
            )

            running_loss += loss.item() * src.size(0)

    return running_loss / len(dataloader.dataset)

In [ ]:
encoder = EncoderRNN(vocab_size=vocab_size, embedding_dim=128, hidden_dim=256)
decoder = DecoderRNN(vocab_size=vocab_size, embedding_dim=128, hidden_dim=256)

seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)

criterion_s2s = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer_s2s = optim.Adam(seq2seq_model.parameters(), lr=0.001)

num_epochs_s2s = 10
best_val_loss_s2s = np.inf
best_epoch_s2s = 0

seq2seq_history = {
    "train_loss": [],
    "val_loss": []
}

seq2seq_save_path = os.path.join(MODEL_DIR, "Seq2Seq_best.pt")

for epoch in range(1, num_epochs_s2s + 1):
    train_loss_s2s = train_seq2seq(
        seq2seq_model,
        train_loader_s2s,
        criterion_s2s,
        optimizer_s2s,
        device,
        teacher_forcing_ratio=0.5
    )

    val_loss_s2s = evaluate_seq2seq(
        seq2seq_model,
        val_loader_s2s,
        criterion_s2s,
        device
    )

    seq2seq_history["train_loss"].append(train_loss_s2s)
    seq2seq_history["val_loss"].append(val_loss_s2s)

    print(
        f"Epoch {epoch:02d}/{num_epochs_s2s} | "
        f"Train loss: {train_loss_s2s:.4f} | "
        f"Val loss: {val_loss_s2s:.4f}"
    )

    if val_loss_s2s < best_val_loss_s2s:
        best_val_loss_s2s = val_loss_s2s
        best_epoch_s2s = epoch
        torch.save(seq2seq_model.state_dict(), seq2seq_save_path)

plt.figure(figsize=(7, 5))
plt.plot(seq2seq_history["train_loss"], label="Train loss")
plt.plot(seq2seq_history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Seq2Seq loss")
plt.legend()
plt.tight_layout()

seq2seq_curve_path = os.path.join(RESULTS_DIR, "Seq2Seq_curves.png")
plt.savefig(seq2seq_curve_path, dpi=300)
plt.show()

print("Meilleure validation loss Seq2Seq :", best_val_loss_s2s)
print("Meilleur epoch Seq2Seq :", best_epoch_s2s)
print("Modèle Seq2Seq sauvegardé :", seq2seq_save_path)

## 14. Décodage Seq2Seq

In [ ]:
def seq2seq_greedy_decode(model, src, max_length=25):
    model.eval()

    src = src.unsqueeze(0).to(device)

    generated = []

    with torch.no_grad():
        _, hidden = model.encoder(src)

        input_token = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)

        for _ in range(max_length):
            logits, hidden = model.decoder(input_token, hidden)
            next_token = logits.argmax(dim=-1).item()

            if next_token == EOS_IDX:
                break

            generated.append(next_token)
            input_token = torch.tensor([[next_token]], dtype=torch.long).to(device)

    return generated


def seq2seq_beam_decode(model, src, max_length=25, beam_width=3):
    model.eval()

    src = src.unsqueeze(0).to(device)

    with torch.no_grad():
        _, hidden = model.encoder(src)

        beams = [([SOS_IDX], 0.0, hidden)]

        for _ in range(max_length):
            candidates = []

            for seq, score, hidden_state in beams:
                input_token = torch.tensor([[seq[-1]]], dtype=torch.long).to(device)
                logits, new_hidden = model.decoder(input_token, hidden_state)

                log_probs = torch.log_softmax(logits[0, -1], dim=0)
                top_values, top_indices = torch.topk(log_probs, beam_width)

                for log_prob, token in zip(top_values, top_indices):
                    new_seq = seq + [token.item()]
                    new_score = score + log_prob.item()
                    candidates.append((new_seq, new_score, new_hidden))

            beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]

        best_sequence = beams[0][0]

    return [token for token in best_sequence if token not in [SOS_IDX, EOS_IDX, PAD_IDX]]

In [ ]:
seq2seq_model.load_state_dict(torch.load(seq2seq_save_path, map_location=device))
seq2seq_model = seq2seq_model.to(device)

src_example, decoder_input_example, target_example = test_dataset_s2s[0]

greedy_s2s = seq2seq_greedy_decode(seq2seq_model, src_example, max_length=25)
beam_s2s = seq2seq_beam_decode(seq2seq_model, src_example, max_length=25, beam_width=3)

print("Séquence d'entrée :")
print(tokens_to_midi_notes(src_example.tolist(), idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(src_example.tolist(), idx_to_note)])

print("\nVraie continuation :")
print(tokens_to_midi_notes(target_example.tolist(), idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(target_example.tolist(), idx_to_note)])

print("\nContinuation générée avec greedy decoding :")
print(tokens_to_midi_notes(greedy_s2s, idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(greedy_s2s, idx_to_note)])

print("\nContinuation générée avec beam search :")
print(tokens_to_midi_notes(beam_s2s, idx_to_note))
print([midi_pitch_to_name(n) for n in tokens_to_midi_notes(beam_s2s, idx_to_note)])

## 15. Évaluation finale et sauvegarde des exemples générés

In [ ]:
test_loss_best, test_perplexity_best = evaluate_sequence(
    best_sequence_model,
    test_loader_seq,
    criterion_seq,
    device
)

print("Meilleur modèle RNN/LSTM/GRU :", best_sequence_row["model_name"])
print(f"Test loss : {test_loss_best:.4f}")
print(f"Test perplexity : {test_perplexity_best:.4f}")

seq2seq_val_loss_final = evaluate_seq2seq(
    seq2seq_model,
    val_loader_s2s,
    criterion_s2s,
    device
)

print("\nSeq2Seq validation loss :", round(seq2seq_val_loss_final, 4))

generated_examples_path = os.path.join(RESULTS_DIR, "generated_music_examples.txt")

with open(generated_examples_path, "w") as f:
    f.write("Partie III - Exemples de génération musicale\n\n")

    f.write("Meilleur modèle séquentiel : " + str(best_sequence_row["model_name"]) + "\n")
    f.write("Test loss : " + str(test_loss_best) + "\n")
    f.write("Test perplexity : " + str(test_perplexity_best) + "\n\n")

    f.write("Seed test MIDI :\n")
    f.write(str(tokens_to_midi_notes(seed_tokens, idx_to_note)) + "\n\n")

    f.write("Génération greedy :\n")
    f.write(str(tokens_to_midi_notes(greedy_tokens, idx_to_note)) + "\n")
    f.write(str([midi_pitch_to_name(n) for n in tokens_to_midi_notes(greedy_tokens, idx_to_note)]) + "\n\n")

    f.write("Génération beam search :\n")
    f.write(str(tokens_to_midi_notes(beam_tokens, idx_to_note)) + "\n")
    f.write(str([midi_pitch_to_name(n) for n in tokens_to_midi_notes(beam_tokens, idx_to_note)]) + "\n\n")

    f.write("Seq2Seq - entrée :\n")
    f.write(str(tokens_to_midi_notes(src_example.tolist(), idx_to_note)) + "\n\n")

    f.write("Seq2Seq - vraie continuation :\n")
    f.write(str(tokens_to_midi_notes(target_example.tolist(), idx_to_note)) + "\n\n")

    f.write("Seq2Seq - génération greedy :\n")
    f.write(str(tokens_to_midi_notes(greedy_s2s, idx_to_note)) + "\n\n")

    f.write("Seq2Seq - génération beam search :\n")
    f.write(str(tokens_to_midi_notes(beam_s2s, idx_to_note)) + "\n")

print("Exemples générés sauvegardés :", generated_examples_path)

## 16. Inspection PyTorch du meilleur modèle

In [ ]:
print("Paramètres nommés du meilleur modèle :")
for name, param in best_sequence_model.named_parameters():
    print(name, param.shape, "trainable =", param.requires_grad)

print("\nstate_dict du meilleur modèle :")
for key, value in best_sequence_model.state_dict().items():
    print(key, value.shape)

total_params_best = count_trainable_parameters(best_sequence_model)

print("\nNombre total de paramètres entraînables :", total_params_best)

model_device = next(best_sequence_model.parameters()).device
print("Device utilisé par le modèle :", model_device)

## 17. Squelette d'analyse à compléter

### Comparaison RNN simple vs LSTM vs GRU

Le RNN simple sert de modèle récurrent de base.  
Le LSTM et le GRU ajoutent des mécanismes de portes pour mieux retenir l'information sur des séquences longues.

À compléter avec les résultats :  
- Loss validation RNN : ...
- Loss validation LSTM : ...
- Loss validation GRU : ...
- Meilleur modèle : ...

### Stabilité de l'entraînement

À compléter :  
- Le modèle le plus stable semble être : ...
- Les courbes montrent que : ...

### Effet du gradient clipping

Le gradient clipping limite la norme des gradients.  
Il aide à éviter les gradients explosifs dans les modèles récurrents.

À compléter :  
- Effet observé pendant l'entraînement : ...

### Analyse de la perplexité

La perplexité mesure l'incertitude du modèle dans la prédiction de la prochaine note.  
Plus elle est faible, plus le modèle est confiant et précis.

À compléter :  
- Perplexité finale du meilleur modèle : ...
- Interprétation : ...

### Qualité des séquences générées

À compléter :  
- Les notes générées semblent-elles cohérentes ?
- Observe-t-on des répétitions ?
- La génération garde-t-elle une structure musicale ?

### Différence entre prédiction de prochaine note et Seq2Seq

La prédiction de prochaine note apprend à prédire chaque note suivante.  
Le Seq2Seq apprend plutôt à transformer un début de phrase musicale en continuation.

À compléter :  
- Différence observée dans les sorties : ...

### Effet du greedy decoding

Le greedy decoding choisit toujours la note la plus probable.  
Il est simple, mais peut produire des séquences répétitives.

À compléter :  
- Résultat observé : ...

### Effet du beam search

Le beam search garde plusieurs hypothèses et peut produire une continuation plus globale.  
Il est plus coûteux que le greedy decoding.

À compléter :  
- Résultat observé : ...

### Limites du projet

Limites possibles :  
- Les notes sont représentées uniquement par leur pitch
- Les durées, vélocités et silences ne sont pas modélisés
- Le mode rapide utilise seulement une partie du dataset
- Les séquences longues restent difficiles à apprendre
- L'évaluation qualitative reste subjective

### Réponse à la question de synthèse

Dans quelle mesure les architectures récurrentes permettent-elles de modéliser efficacement une séquence réelle, et comment justifier le passage d'un RNN simple vers un LSTM/GRU puis vers un schéma encodeur-décodeur pour une tâche de génération musicale ?

Réponse à compléter :

Les architectures récurrentes permettent de modéliser une séquence réelle car elles traitent les éléments dans l'ordre et conservent une mémoire des éléments précédents. Pour la musique, cette mémoire est importante car une note dépend souvent du contexte mélodique précédent.

Un RNN simple constitue une première approche, mais il devient instable sur des séquences longues à cause des gradients qui disparaissent ou explosent. Les LSTM et GRU améliorent cette situation grâce à leurs portes, qui contrôlent mieux l'information conservée ou oubliée. Le passage vers un modèle Seq2Seq se justifie lorsque l'objectif n'est plus seulement de prédire la prochaine note, mais de générer une continuation complète à partir d'un début de phrase musicale.